# Chapter 03. Structured Outputs로 구조화된 인과 분석 결과 받기

## 학습 목표

- Structured Outputs의 원리와 필요성을 이해한다
- Pydantic 모델과 JSON Schema의 관계를 이해한다
- 핵심 Pydantic 모델(Variables, DatasetAnalysis, MethodInfo)을 분석한다
- `client.beta.chat.completions.parse()`로 구조화된 변수 식별을 구현한다

## 환경 설정

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional, Union

# OpenAI API 키 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## 1. Pydantic 모델 기초

In [2]:
# ============================================================
# Pydantic 모델 기초 — 모델 구조를 이해하기 위한 준비
# ============================================================

class SimpleVariable(BaseModel):
    """가장 기본적인 변수 식별 모델이다."""
    name: str = Field(description="변수 이름")
    role: str = Field(description="변수 역할: treatment, outcome, covariate")
    data_type: str = Field(description="데이터 타입: binary, continuous, categorical")

# Pydantic 모델의 핵심 기능
var = SimpleVariable(name="format_ol", role="treatment", data_type="binary")

# 1) 딕셔너리 변환
print("dict:", var.model_dump())

# 2) JSON 직렬화
print("JSON:", var.model_dump_json(indent=2))

# 3) JSON Schema 자동 생성 (OpenAI가 이것을 사용한다)
print("JSON Schema:", json.dumps(SimpleVariable.model_json_schema(), indent=2))

dict: {'name': 'format_ol', 'role': 'treatment', 'data_type': 'binary'}
JSON: {
  "name": "format_ol",
  "role": "treatment",
  "data_type": "binary"
}
JSON Schema: {
  "description": "\uac00\uc7a5 \uae30\ubcf8\uc801\uc778 \ubcc0\uc218 \uc2dd\ubcc4 \ubaa8\ub378\uc774\ub2e4.",
  "properties": {
    "name": {
      "description": "\ubcc0\uc218 \uc774\ub984",
      "title": "Name",
      "type": "string"
    },
    "role": {
      "description": "\ubcc0\uc218 \uc5ed\ud560: treatment, outcome, covariate",
      "title": "Role",
      "type": "string"
    },
    "data_type": {
      "description": "\ub370\uc774\ud130 \ud0c0\uc785: binary, continuous, categorical",
      "title": "Data Type",
      "type": "string"
    }
  },
  "required": [
    "name",
    "role",
    "data_type"
  ],
  "title": "SimpleVariable",
  "type": "object"
}


## 2. 핵심 모델 재구현

In [4]:
# ============================================================
# 핵심 Pydantic 모델을 OpenAI SDK용으로 재구현한다
# ============================================================

class Variables(BaseModel):
    """인과 분석에서 식별된 변수 구조이다."""
    treatment_variable: str = Field(description="처치 변수명")
    treatment_variable_type: str = Field(description="처치 변수 타입: binary, continuous, categorical")
    outcome_variable: str = Field(description="결과 변수명")
    covariates: List[str] = Field(default_factory=list, description="통제할 공변량 리스트")
    instrument_variable: Optional[str] = Field(None, description="도구변수 (IV 분석용)")
    time_variable: Optional[str] = Field(None, description="시간 변수 (DiD 분석용)")
    running_variable: Optional[str] = Field(None, description="배정 변수 (RDD 분석용)")
    cutoff_value: Optional[float] = Field(None, description="임계값 (RDD 분석용)")
    is_rct: bool = Field(description="무작위 대조 실험 여부")
    reasoning: str = Field(description="변수 식별의 근거")

class MethodRecommendation(BaseModel):
    """방법론 추천 결과이다."""
    selected_method: str = Field(description="추천 방법론: diff_in_means, linear_regression, did, iv, rdd, psm, psw, backdoor")
    method_name: str = Field(description="방법론 한국어 이름")
    confidence: float = Field(description="추천 신뢰도 (0~1)")
    justification: str = Field(description="추천 근거")
    assumptions: List[str] = Field(description="핵심 가정 리스트")
    alternative_methods: List[str] = Field(default_factory=list, description="대안 방법론")

class RCTCheck(BaseModel):
    """RCT 여부 판단 결과이다."""
    is_rct: bool = Field(description="무작위 실험인지 여부")
    reasoning: str = Field(description="판단 근거")
    confidence: float = Field(description="판단 신뢰도 (0~1)")

print("핵심 모델 3개 정의 완료:")
print("  - Variables: 인과 변수 구조")
print("  - MethodRecommendation: 방법론 추천")
print("  - RCTCheck: RCT 여부 판단")

핵심 모델 3개 정의 완료:
  - Variables: 인과 변수 구조
  - MethodRecommendation: 방법론 추천
  - RCTCheck: RCT 여부 판단


## 3. Structured Outputs로 변수 식별

In [5]:
# ============================================================
# Structured Outputs로 실제 데이터에서 변수를 식별한다
# ============================================================

# 데이터 로드
df_hospital = pd.read_csv("./dataset/hospital_treatment.csv")
df_wage = pd.read_csv("./dataset/wage.csv")

# 데이터셋 정보 생성 함수
def get_dataset_info(df, name):
    """데이터셋 정보를 LLM에 전달할 문자열로 변환한다."""
    info = f"데이터셋: {name}\n"
    info += f"행: {len(df)}, 열: {len(df.columns)}\n"
    info += f"컬럼: {list(df.columns)}\n"
    info += f"타입:\n{df.dtypes.to_string()}\n"
    info += f"샘플(3행):\n{df.head(3).to_string()}"
    return info

# --- 예제 1: hospital_treatment.csv ---
print("=== 예제 1: 병원 치료 데이터 ===")
response1 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "너는 인과추론 전문가이다. 데이터와 질문에서 인과 분석에 필요한 변수를 식별한다."},
        {"role": "user", "content": f"""다음 데이터에서 변수를 식별해줘.

질문: "새로운 치료법(treatment)이 환자의 회복률(recovery)에 미치는 효과는?"

{get_dataset_info(df_hospital, "hospital_treatment.csv")}
"""}
    ],
    response_format=Variables
)

vars1 = response1.choices[0].message.parsed
print(f"처치변수: {vars1.treatment_variable}")
print(f"결과변수: {vars1.outcome_variable}")
print(f"RCT 여부: {vars1.is_rct}")
print(f"공변량: {vars1.covariates}")
print(f"근거: {vars1.reasoning}")
print()

=== 예제 1: 병원 치료 데이터 ===
처치변수: treatment
결과변수: days
RCT 여부: False
공변량: ['hospital', 'severity']
근거: 치료법(treatment)과 회복률(days)의 상관관계를 분석하기 위해 치료법은 이진 변수로 간주되며, 이는 치료를 받았는지 여부를 나타냄. 회복률을 days로 정의하고, 병원(hospital)과 환자의 중증도(severity)를 통제 변수로 사용함.



In [6]:
# --- 예제 2: wage.csv ---
print("=== 예제 2: 임금 데이터 ===")
response2 = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "너는 인과추론 전문가이다. 데이터와 질문에서 인과 분석에 필요한 변수를 식별한다."},
        {"role": "user", "content": f"""다음 데이터에서 변수를 식별해줘.

질문: "교육 수준(educ)이 시간당 임금(wage)에 미치는 인과적 효과는? 출생 분기(quarter of birth)를 도구변수로 사용할 수 있다."

{get_dataset_info(df_wage, "wage.csv")}
"""}
    ],
    response_format=Variables
)

vars2 = response2.choices[0].message.parsed
print(f"처치변수: {vars2.treatment_variable}")
print(f"결과변수: {vars2.outcome_variable}")
print(f"도구변수: {vars2.instrument_variable}")
print(f"RCT 여부: {vars2.is_rct}")
print(f"근거: {vars2.reasoning}")

=== 예제 2: 임금 데이터 ===
처치변수: educ
결과변수: wage
도구변수: quarter of birth
RCT 여부: False
근거: The treatment variable is 'educ' (education level) as it is being examined for its causal effect on the outcome variable 'wage' (hourly wage). 'quarter of birth' is used as an instrumental variable to address potential endogeneity issues between education and wage. Several covariates are included to control for other factors that might influence wage.


## 4. 방법론 추천 구조화

In [7]:
# ============================================================
# 식별된 변수로 방법론을 추천받는다
# ============================================================
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": 
         """너는 인과추론 방법론 전문가이다.
            사용 가능한 방법론: diff_in_means, linear_regression, difference_in_differences, 
            instrumental_variable, regression_discontinuity_design, propensity_score_matching, 
            propensity_score_weighting, backdoor_adjustment"""},
        {"role": "user", "content": f"""다음 변수 정보로 최적 방법론을 추천해줘.

처치변수: {vars2.treatment_variable} (타입: {vars2.treatment_variable_type})
결과변수: {vars2.outcome_variable}
도구변수: {vars2.instrument_variable}
RCT 여부: {vars2.is_rct}
공변량: {vars2.covariates}
"""}
    ],
    response_format=MethodRecommendation
)

rec = response.choices[0].message.parsed
print("=== 방법론 추천 결과 ===")
print(f"추천 방법론: {rec.selected_method} ({rec.method_name})")
print(f"신뢰도: {rec.confidence:.1%}")
print(f"근거: {rec.justification}")
print(f"핵심 가정: {rec.assumptions}")
print(f"대안: {rec.alternative_methods}")

=== 방법론 추천 결과 ===
추천 방법론: iv (Instrumental Variable)
신뢰도: 85.0%
근거: 주어진 처치변수(educ)에 대한 인과적 영향을 분석하기 위해 도구변수(quarter of birth)를 사용합니다. 이는 교육이 임금을 어떻게 변화시키는지를 알아보기 위한 적절한 접근 방식입니다.
핵심 가정: ['The instrument (quarter of birth) is correlated with the endogenous treatment variable (educ).', 'The instrument affects the outcome (wage) only through its effect on the treatment variable (educ).', 'There are no unmeasured confounders between the treatment variable and the outcome.']
대안: ['linear_regression', 'propensity_score_matching', 'difference_in_differences']
